# 学習（広島）

**この雛形は `docs/ノートブックの作り方.md` の契約そのものである。**
セルを足すときもあの文書に従うこと。

- 値をここに書かない。すべて `config.yaml` から来る
- 処理をここに書かない。`dc5lib` を呼ぶ。学習ループを書き写さない
- 島根の評価は `eval_shimane.ipynb`。ここには入れない

困ったら `python tools/audit_wiring.py --condition <条件名> --show` で、
どの行が config を通っているかを行ごと見られる。

In [ ]:
# ============================================================
# 条件を1か所で決める。**ここ以外に条件名を書かない。**
# 重み・曲線・pkl の場所はすべて dc5lib が条件名から解決する。
# ============================================================
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "dc5lib" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import matplotlib.pyplot as plt
import pandas as pd
import torch

from dc5lib import ckpt, paths
from dc5lib.evaluate import evaluate
from dc5lib.models import build_model_for
from dc5lib.pipeline import build_loaders, seed_everything
from dc5lib.registry import get
from dc5lib.train import fit
from dc5lib.traincfg import train_cfg

CONDITION = "__CONDITION__"
REGION = "Hiroshima"

cond = get(CONDITION)
CFG = train_cfg(CONDITION)
print(cond.display())
print("note:", cond.note or "（なし）")
print("protocol:", cond.protocol, "/ ensemble:", cond.ensemble)
print()
print(CFG.describe())
print()
print("--- config.yaml ---")
print((cond.dir / "config.yaml").read_text(encoding="utf-8"))

In [ ]:
# ============================================================
# 乱数の固定。**これが無いと CFG.seed は何も変えない。**
# 既存56本のうち12本しか呼んでいなかった（ゆらぎ幅.md の 8）。
# cuDNN の非決定性は残るのでビット単位では一致しない。
# ============================================================
seed_everything(CFG.seed)
print("乱数を固定:", CFG.seed, "/ seeded を config に true で書くこと")

In [ ]:
# ============================================================
# pkl → マスク前処理 → 分割 → DataLoader。手順は dc5lib/pipeline.py。
# **近傍は背景タイルからも貼り、検証・テストだけを外している。**
# ============================================================
ld = build_loaders(cond, REGION)

In [ ]:
# ============================================================
# **回す前にここを目で見る。** 違っていたら回さない。
# 違うのは pkl か分割の指定か前処理で、どれも他の条件と比べられなくする。
# 期待値は docs/ノートブックの作り方.md の 2.3 にある。
# ============================================================
EXPECT = {  # 広島・random・random_state=42 のとき
    "n_used": 14816, "n_train": 8889, "n_val": 2963, "n_test": 2964,
    "tiles_fingerprint": "a75ad23bad3e", "fingerprint": "653b8cbe3d28",
}
got = {"n_used": ld.n_used, **{k: ld.split[k] for k in
                              ("n_train", "n_val", "n_test",
                               "tiles_fingerprint", "fingerprint")}}
for k, want in EXPECT.items():
    ok = got[k] == want
    print(f"  {'一致  ' if ok else '**違う**'} {k:<18}{got[k]}"
          f"{'' if ok else f'  （既存条件は {want}）'}")
print("\n※ pkl が違う条件（256px など）や分割を変えた条件では違って当然。"
      "その場合は EXPECT を自分の条件の値に書き換え、"
      "tools/audit_splits.py で他の条件と食い違わないかを測ること。")

In [ ]:
# ============================================================
# モデル。**build_model_for を使う。** build_model(arch) だと
# 3入力目のチャネル数が既定（土地利用の9）になって落ちる。
# ============================================================
model = build_model_for(cond)
n_par = sum(p.numel() for p in model.parameters())
print("入力のチャネル:", cond.in_channels, "/ パラメータ %.2fM" % (n_par / 1e6))
if len(cond.in_channels) >= 3:
    assert model.geo_enc1[0].in_channels == cond.in_channels[2], \
        "3本目の符号化器の入口が config と違う"
    print("3本目の符号化器の入口:", model.geo_enc1[0].in_channels)

In [ ]:
# ============================================================
# 学習。早期終了・曲線の毎エポック書き出し・検証Fでの選択・
# 素性つきの重みの保存は、すべて fit() が引き受ける。
# **ループを書き写さない。** 細工は hooks で入れる
#   hooks={"forward":..., "loss_fn":..., "on_epoch_end":..., "optimizer":...}
# ============================================================
hist = fit(cond, ld, hooks={"model": model, "seeded": True})

In [ ]:
# ============================================================
# 学習曲線。**CSV から読む**（fit が毎エポック flush している）。
# 画面の履歴ではなくファイルを見るので、途中で落ちても描ける。
# ============================================================
df = pd.read_csv(paths.curve_path(CONDITION))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.Epoch, df.Loss)
ax[0].set_title("Loss")
ax[0].set_xlabel("epoch")
ax[1].plot(df.Epoch, df["F1-score"], label="train F")
ax[1].plot(df.Epoch, df["Validation-F1-score"], label="val F")
ax[1].axvline(hist.best_epoch, ls="--", c="gray")
ax[1].set_title("F1")
ax[1].set_xlabel("epoch")
ax[1].legend()
plt.tight_layout()
plt.show()
print(f"{len(df)} 行 / 最良 エポック {hist.best_epoch} の "
      f"検証F {hist.best_val_f1:.4f}")

In [ ]:
# ============================================================
# **重みの素性。** この重みがどの設定・どの分割・どの commit で
# 学習されたかが、重みファイルの中に入っている。
# 以後 ckpt.same_split(A, B) で、pkl を読み直さずに分割を照合できる。
# ============================================================
print(ckpt.describe(cond.weights))

In [ ]:
# ============================================================
# テストでの値。**素のタイル**での画素評価と箇所評価である。
# オーバーラップ推論の箇所評価は analysis/run_overlap_eval.py が担当する
# （展望_再現率をどう上げるか.md の 42〜44 の数字はそちら）。
# ============================================================
res = evaluate(cond, ld, "test", model=model)

## 次にやること

1. `config.yaml` に `seeded: true` と `protocol: v3-base` を書く（学習したので）
2. `eval_shimane.ipynb` で汎化性能を測る
3. オーバーラップ推論の箇所評価
   ```bash
   python analysis/run_overlap_eval.py --region both --conditions <条件名> --record
   ```
4. 和集合に入れるなら、先に分割が一致することを確かめる
   ```bash
   python tools/audit_splits.py --region both --conditions <全構成員>
   ```
5. 出す前に `python tools/check.py` と `python tools/audit_wiring.py`